<div align="center">

# Week 05: Design Principles, Heuristics and Human Error

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-05) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/Week05_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the heuristic evaluation of the clinic app by five evaluators, the course data of stage 5. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import itertools
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The heuristic evaluation of the clinic app by five evaluators: one problem per
# row, the heuristic it breaks, who found it (one digit per evaluator) and the
# severity ratings 0 to 4 of all five evaluators. Course data of stage 5.
CLINIC_EVAL = pd.DataFrame([
    (1, "No booking reference after booking", "H1", "11011", [3, 4, 3, 4, 3]),
    (2, "Past dates and closed days can be chosen", "H5", "10101", [4, 4, 3, 4, 3]),
    (3, "The error message shows only a code", "H9", "01101", [3, 3, 3, 3, 3]),
    (4, "Departments named in medical terms", "H2", "10010", [3, 3, 2, 3, 3]),
    (5, "Cancel next to Confirm, equal in size", "H5", "11111", [4, 4, 4, 3, 4]),
    (6, "No way back from the summary screen", "H3", "01010", [3, 3, 4, 3, 3]),
    (7, "Appointment, booking and visit mean the same", "H4", "00100",
     [2, 2, 1, 2, 2]),
    (8, "Patient number typed from memory", "H6", "11001", [3, 3, 3, 3, 3]),
    (9, "Time chips only 7 mm wide", "H5", "10100", [3, 4, 3, 3, 4]),
    (10, "Help is a PDF of 48 pages", "H10", "00001", [2, 1, 2, 1, 2]),
    (11, "Home screen with 14 tiles of equal weight", "H8", "01000",
     [2, 2, 3, 2, 2]),
    (12, "No shortcut to book the last visit again", "H7", "00010",
     [2, 2, 2, 2, 2]),
    (13, "The session ends without a warning", "H1", "10001", [3, 3, 3, 3, 3]),
    (14, "Continue on the left at the kiosk, on the right in the app", "H4",
     "00100", [2, 3, 2, 3, 3]),
    (15, "SMS reminder without the department or room", "H6", "10000",
     [2, 3, 2, 2, 3]),
    (16, "Confirmation e-mail shows the time in UTC", "H2", "01001",
     [3, 3, 3, 2, 3]),
], columns=["problem", "description", "heuristic", "found_by", "ratings"])
print("Setup complete. Running in Colab:", IN_COLAB,
      "| problems of the clinic evaluation:", len(CLINIC_EVAL))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 5.1": (lambda: int(np.array([[1, 0, 1, 1], [0, 0, 1, 0], [1, 1, 1, 0]]).sum(axis=0).max()), 1e-9),
    "Exercise 5.2": (lambda: float(np.mean(np.array([[1, 0, 1, 1], [0, 0, 1, 0], [1, 1, 1, 0]]))), 1e-9),
    "Exercise 5.3": (lambda: int((np.array([[1, 0, 1, 1], [0, 0, 1, 0], [1, 1, 1, 0]]).sum(axis=0) == 1).sum()), 1e-9),
    "Exercise A": (lambda: 2 / 6, 1e-6),
    "Exercise B": (lambda: 5, 1e-9),
    "Exercise C": (lambda: 1 - (1 - 0.01) ** 30, 1e-6),
    "Exercise D": (lambda: 0.10 * 0.5 * 0.59, 1e-6),
    "Exercise E": (lambda: float(STAGE5["evaluators for 90 percent"]), 1e-9),
}

## Python step 5: Matrices of zeros and ones

A heuristic evaluation produces a matrix with one row for each evaluator and one column for each problem. A one means that the evaluator found the problem and a zero that the evaluator missed it. This step builds such a matrix with NumPy &#91;[2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#ref-2)&#93;, sums it along its rows and columns and follows how the found problems accumulate as evaluators are added. It ends by sorting a pandas table by several columns, the way problems are ranked by severity &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#ref-1)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "CLINIC_EVAL" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-05/"
               "NB05_heuristics_and_error.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### A matrix of who found what

In [ ]:
# Python practice: a small example for learning the language
M = np.array([[1, 0, 1, 1, 0, 0],    # evaluator 1
              [0, 1, 1, 0, 0, 1],    # evaluator 2
              [1, 0, 1, 0, 1, 0],    # evaluator 3
              [0, 0, 1, 1, 0, 0]])   # evaluator 4
print("shape (evaluators, problems):", M.shape)
print("problems found by each evaluator:", M.sum(axis=1))
print("evaluators who found each problem:", M.sum(axis=0))
print("share of all cells that are ones:", M.mean())

`M.shape` gives the number of rows and columns. The argument `axis=1` sums along each row, so it counts the problems of each evaluator, and `axis=0` sums down each column, so it counts the evaluators of each problem. The mean of all cells is the share of evaluator and problem pairs in which the problem was found.

### Sums along rows and columns

In [ ]:
# Python practice: a small example for learning the language
per_problem = M.sum(axis=0)
print("found by nobody:", np.sum(per_problem == 0))
print("found by exactly one evaluator:", np.sum(per_problem == 1))
print("found by all four:", np.sum(per_problem == M.shape[0]))
print("problems found at all:", np.sum(per_problem > 0), "of", M.shape[1])

A comparison such as `per_problem == 1` returns an array of `True` and `False`, and `np.sum` counts the values `True`. Problems found by exactly one evaluator matter later in the week: Many of them signal that further problems were found by nobody.

### The first evaluators and the problems they find together

In [ ]:
# Python practice: a small example for learning the language
for k in range(1, M.shape[0] + 1):
    found = M[:k].any(axis=0)          # found by at least one of the first k
    print(f"first {k} evaluators: {found.sum()} problems, {found.mean():.2f}")
rng = np.random.default_rng(5)
order = rng.permutation(M.shape[0])
print("a random order of the evaluators:", order)
print("the same matrix in that order:", M[order].sum(axis=1))

`M[:k]` keeps the first `k` rows, and `any(axis=0)` asks for each column whether at least one of them is one. The loop therefore shows how the found problems accumulate. `rng.permutation` shuffles the numbers of the rows, and `M[order]` rearranges the rows in that order.

### Sorting a table by several columns

In [ ]:
# Python practice: a small example for learning the language
tab = pd.DataFrame({"problem": [1, 2, 3, 4, 5, 6],
                    "severity": [3.0, 2.2, 3.6, 3.0, 1.8, 2.2],
                    "found_by": M.sum(axis=0)})
ranked = tab.sort_values(["severity", "found_by"], ascending=[False, False])
print(ranked.to_string(index=False))

`sort_values` with a list of columns sorts by the first one and breaks ties with the second. The list `ascending=[False, False]` sorts both from the largest value down, so the most severe problems come first, and among equally severe ones the problem that more evaluators found.

**Quick check.** Why can the share of found problems never fall when an evaluator is added?

<details><summary>Answer</summary>

A problem counts as found as soon as one evaluator has found it. Adding a row can turn zeros of the column test into ones but never the reverse, so the share stays or grows.

</details>

### Exercises for Python step 5

**Exercise 5.1.** Three evaluators produced the matrix `[[1, 0, 1, 1], [0, 0, 1, 0], [1, 1, 1, 0]]`. Store the largest number of evaluators who found the same problem in `most_found`.

In [ ]:
most_found = None   # your answer

check("Exercise 5.1", most_found)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
E = np.array([[1, 0, 1, 1], [0, 0, 1, 0], [1, 1, 1, 0]])
most_found = int(E.sum(axis=0).max())

check("Exercise 5.1", most_found)

**Exercise 5.2.** Store the share of the cells of the same matrix that are ones in `share_ones`.

In [ ]:
share_ones = None   # your answer

check("Exercise 5.2", share_ones)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
E = np.array([[1, 0, 1, 1], [0, 0, 1, 0], [1, 1, 1, 0]])
share_ones = float(E.mean())

check("Exercise 5.2", share_ones)

**Exercise 5.3.** Store the number of problems of the same matrix that exactly one evaluator found in `found_once`.

In [ ]:
found_once = None   # your answer

check("Exercise 5.3", found_once)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
E = np.array([[1, 0, 1, 1], [0, 0, 1, 0], [1, 1, 1, 0]])
found_once = int((E.sum(axis=0) == 1).sum())

check("Exercise 5.3", found_once)

## 1. Heuristic evaluation: Severity and agreement &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#ref-4)&#93;

*Lecture: [Heuristic evaluation](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#heuristic-evaluation)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "ranked" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-05/"
               "NB05_heuristics_and_error.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A heuristic evaluation ends with a list of problems found by several evaluators, each rated for severity. Python ranks the problems by their mean severity and measures how far the evaluators agreed. Agreement is hard to judge by eye, but a few lines of code compute it for every pair of evaluators.

**From the formulas to the code.** The sets of problems found by two evaluators $P_i$ and $P_j$ are `a` and `b`, and their agreement $|P_i \cap P_j| \,/\, |P_i \cup P_j|$ is `agreement(a, b)`. The mean of all pairs is `mean_agreement(M)` for a matrix `M` with one row per evaluator.

In [ ]:
def agreement(a, b):
    """Problems found by both evaluators, divided by those found by either."""
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else 1.0

def mean_agreement(M):
    """Mean agreement over all pairs of rows of a 0/1 matrix."""
    sets = [set(np.flatnonzero(row)) for row in np.asarray(M)]
    pairs = itertools.combinations(sets, 2)
    return float(np.mean([agreement(a, b) for a, b in pairs]))

print("worked example:", agreement({1, 2, 5, 8}, {2, 5, 9}))
ev = CLINIC_EVAL.copy()
ev["mean severity"] = ev["ratings"].apply(np.mean)
ev["found by"] = ev["found_by"].apply(lambda s: s.count("1"))
M5 = np.array([[int(s[i]) for s in ev["found_by"]] for i in range(5)])
print("clinic app: mean agreement of the five evaluators",
      round(mean_agreement(M5), 3))
print(ev.sort_values(["mean severity", "found by"], ascending=False)
        .head(6)[["problem", "description", "heuristic", "mean severity",
                  "found by"]].to_string(index=False))

**Python note.** `set` keeps each problem once, `&` forms the problems in both sets and `|` those in either. `itertools.combinations(sets, 2)` returns every pair of evaluators once, and `np.flatnonzero` returns the positions of the ones in a row.

**Exercise A.** One evaluator found the problems 1, 3, 4, 7 and 9, and another found 3, 5 and 7. Store their agreement in `agree_b`.

In [ ]:
agree_b = None   # your answer

check("Exercise A", agree_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
agree_b = agreement({1, 3, 4, 7, 9}, {3, 5, 7})

check("Exercise A", agree_b)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_HUNT`. Your finds then become the first row of a class of twelve evaluators. The other eleven are simulated with detection probabilities that are assumptions of this Colab notebook.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_HUNT = """
"""
HUNT_P = np.array([0.55, 0.35, 0.60, 0.45, 0.70, 0.50, 0.30, 0.65, 0.75, 0.25])

def simulated_hunt_class(n=11, seed=RANDOM_STATE):
    """Finds of n simulated evaluators for the ten problems of Part A."""
    rng = np.random.default_rng(seed + 1)
    return (rng.random((n, len(HUNT_P))) < HUNT_P).astype(int)

others = simulated_hunt_class()
if MY_HUNT.strip():
    hunt = pd.read_csv(io.StringIO(MY_HUNT.strip()))
    mine = hunt.sort_values("problem")["found"].to_numpy().astype(int)
    source = "your own finds and 11 simulated classmates"
else:
    mine = simulated_hunt_class(1, seed=7)[0]
    source = "a simulated evaluator and 11 simulated classmates"
class_M = np.vstack([mine, others])
print("Data:", source, "| matrix", class_M.shape)
print("your finds:          ", mine)
print("share of the class:  ", class_M.mean(axis=0).round(2))
print("mean agreement of the class:", round(mean_agreement(class_M), 3))

**Python note.** `np.vstack` stacks your row on top of the eleven simulated rows. `astype(int)` turns the values `True` and `False` of a comparison into ones and zeros.

In [ ]:
# Figure: share of the class that found each problem
fig, ax = plt.subplots(figsize=(8, 3.4))
problems = np.arange(1, len(HUNT_P) + 1)
share = class_M.mean(axis=0)
ax.bar(problems, share, color="#5A4BB5", label="share of the class")
ax.plot(problems[mine == 1], share[mine == 1], "o", color="#C98510",
        label="found by you")
ax.set_xticks(problems, [f"H{k}" for k in problems])
ax.set_xlabel("problem, named by its heuristic")
ax.set_ylabel("share of evaluators who found it")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Python note.** Indexing with a condition, as in `problems[mine == 1]`, keeps only the problems that you found, so the orange dots mark your finds on the bars of the class.

**Observe and reflect.** Which problems did most of the class find, and which did few find? Is the agreement of the class closer to 5 or to 65 percent, the range reported by Hertzum and Jacobsen?

## 2. How many evaluators? The discovery curve &#91;[8](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#ref-8)&#93;

*Lecture: [How many evaluators?](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#how-many-evaluators)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "class_M" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-05/"
               "NB05_heuristics_and_error.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A team must decide how many evaluators to pay before it knows how many problems exist. The model of Nielsen and Landauer answers this from one number, the probability that one evaluator finds a problem. Python estimates that number from a matrix and corrects it for small samples. It then compares the model with the discovery of the class.

**From the formulas to the code.** The probability that one evaluator finds a problem $\lambda$ is `lam`. The share found by $n$ evaluators $1 - (1 - \lambda)^n$ is `found_share(lam, n)`, and the evaluators for a goal $\ln(1 - g) / \ln(1 - \lambda)$ are `evaluators_for(g, lam)`. The adjustment of Lewis is `lewis_adjust(M)`, with the share of problems found once $GT_{adj}$ as `gt`.

In [ ]:
def found_share(lam, n):
    """Share of all problems found by n evaluators."""
    return 1 - (1 - np.asarray(lam, float)) ** np.asarray(n, float)

def evaluators_for(goal, lam):
    """Smallest whole number of evaluators that reaches the goal share."""
    return int(math.ceil(math.log(1 - goal) / math.log(1 - lam) - 1e-9))

def lewis_adjust(M):
    """Estimate of lambda from a 0/1 matrix and its adjustment for few rows."""
    M = np.asarray(M)
    M = M[:, M.sum(axis=0) > 0]               # only problems found at all
    n = M.shape[0]
    lam = M.mean()
    gt = np.mean(M.sum(axis=0) == 1)          # share of problems found once
    adj = 0.5 * (lam - 1 / n) * (1 - 1 / n) + 0.5 * lam / (1 + gt)
    return lam, gt, adj

print("worked example, lambda = 0.31:",
      f"five evaluators find {float(found_share(0.31, 5)):.3f},",
      f"90 percent needs {evaluators_for(0.90, 0.31)} evaluators")

**Python note.** `math.ceil` rounds up, because the fraction of an evaluator cannot be hired. Subtracting `1e-9` before rounding prevents a value such as 4.000000001, a rounding error of the computer, from becoming 5.

**Exercise B.** Store the number of evaluators needed to find 80 percent of the problems with $\lambda = 0.31$ in `n_80`.

In [ ]:
n_80 = None   # your answer

check("Exercise B", n_80)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_80 = evaluators_for(0.80, 0.31)

check("Exercise B", n_80)

In [ ]:
lam_class = float(class_M.mean())
lam3, gt3, adj3 = lewis_adjust(class_M[:3])
print(f"lambda from all {len(class_M)} evaluators: {lam_class:.3f}")
print(f"from the first 3 only: {lam3:.3f}, adjusted {adj3:.3f} (GT_adj {gt3:.2f})")
rng = np.random.default_rng(RANDOM_STATE)
curves = []
for _ in range(200):
    rows = class_M[rng.permutation(len(class_M))]
    curves.append([rows[:k].any(axis=0).mean() for k in range(1, len(rows) + 1)])
observed = np.mean(curves, axis=0)
print("observed discovery:", observed.round(2))

**Python note.** The observed curve averages 200 random orders of the evaluators, because the order in which they happened to join changes the curve. The first three evaluators stand for a small study, such as a team that has only three evaluators.

In [ ]:
# Figure: the discovery curve of the class
k = np.arange(1, len(class_M) + 1)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(k, observed, "o", color="#C98510", label="observed, mean of 200 orders")
ax.plot(k, found_share(lam_class, k), color="#5A4BB5",
        label=f"model, lambda = {lam_class:.2f}")
ax.plot(k, found_share(lam3, k), "--", color="#B53F2E",
        label=f"from 3 evaluators, {lam3:.2f}")
ax.plot(k, found_share(adj3, k), ":", color="#2E8B57",
        label=f"from 3, adjusted, {adj3:.2f}")
ax.set_xlabel("number of evaluators n")
ax.set_ylabel("share of problems found")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python note.** Each line style marks one estimate. The solid line uses all evaluators, the dashed line the first three and the dotted line their adjustment. The legend writes each estimate into its label with an f-string.

**Explore.** Move the sliders: The panel draws the discovery curve for a probability $\lambda$ and marks the number of evaluators that reaches the goal. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(lam=0.31, goal=0.9):
    n_goal = evaluators_for(goal, lam)
    n = np.arange(1, 21)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(n, found_share(lam, n), "o-", color="#5A4BB5")
    ax.axhline(goal, ls=":", color="#585E6E")
    ax.axvline(n_goal, ls="--", color="#C98510")
    ax.set_xlabel("number of evaluators")
    ax.set_ylabel("share found")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
    print(f"lambda {lam:.2f}: {n_goal} evaluators for {100 * goal:.0f} percent")

slider_lam = widgets.FloatSlider(value=0.31, min=0.05, max=0.6, step=0.01,
                                 description="lambda")
slider_goal = widgets.FloatSlider(value=0.9, min=0.5, max=0.99, step=0.01,
                                  description="goal")
widgets.interact(explore, lam=slider_lam, goal=slider_goal);

**Python note.** `FloatSlider` returns decimal numbers, unlike `IntSlider`. The library ipywidgets calls `explore` again each time a slider moves.

**Observe and reflect.** How far above the observed curve does the estimate from three evaluators lie? Does the adjustment of Lewis bring it closer?

## 3. Slips and mode errors: Your infusion pump &#91;[10](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#ref-10)&#93;

*Lecture: [Human error: Slips, lapses, mistakes and modes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#human-error)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "observed" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-05/"
               "NB05_heuristics_and_error.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A device that is programmed many times a day turns a small error rate into many errors a year. Python turns the rates of the two pumps of the interactive lab into errors per year of a ward. It also computes the probability of at least one error in a day, and a rate alone hides both numbers.

**From the formulas to the code.** The error rate $p$ is `p` and the number of uses $N$ is `n`. The expected number of errors $N p$ is `expected_errors(p, n)`, and the probability of at least one error $1 - (1 - p)^N$ is `p_at_least_one(p, n)`.

In [ ]:
def expected_errors(p, n):
    """Expected number of errors in n uses with the error rate p."""
    return n * np.asarray(p, float)

def p_at_least_one(p, n):
    """Probability of at least one error in n independent uses."""
    return 1 - (1 - np.asarray(p, float)) ** n

print("worked example, p = 0.02 and 12 programmings a day:")
print(f"  errors per year: {float(expected_errors(0.02, 12 * 365)):.1f}")
print(f"  at least one error in a day: {float(p_at_least_one(0.02, 12)):.3f}")

**Python note.** Both functions accept arrays, so the same call computes a whole table of error rates or numbers of uses. The second formula assumes that the uses are independent: An error in one use does not change the chance of an error in the next.

**Exercise C.** A pump is programmed 30 times a day, and each programming ends in a mode error with the probability 0.01. Store the probability of at least one mode error in a day in `p_day`.

In [ ]:
p_day = None   # your answer

check("Exercise C", p_day)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_day = 1 - (1 - 0.01) ** 30

check("Exercise C", p_day)

Part C of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_PUMP`. Left empty, the variable makes the code use a simulated class of 24 students. Their error rates and times are assumptions of this Colab notebook.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_PUMP = """
"""

def simulated_pump_class(n_students=24, seed=RANDOM_STATE):
    """Ten orders per design; half start in the volume mode."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        for block, (design, p_miss) in enumerate([("subtle", 0.22),
                                                  ("salient", 0.04)], 1):
            modes = rng.permutation(["rate"] * 5 + ["volume"] * 5)
            for trial, start in enumerate(modes, 1):
                rate = int(rng.choice([25, 40, 60, 75, 80, 100, 120, 150]))
                mode_error = int(start == "volume" and rng.random() < p_miss)
                value_error = int(rng.random() < 0.02)
                rt = rng.lognormal(np.log(5.5 if start == "rate" else 6.5), 0.2)
                rows.append([student, block, design, trial, rate, start,
                             mode_error, value_error, round(1000 * rt)])
    return pd.DataFrame(rows, columns=["student", "block", "design", "trial",
                                       "ordered", "start_mode", "mode_error",
                                       "value_error", "rt_ms"])

if MY_PUMP.strip():
    pump = pd.read_csv(io.StringIO(MY_PUMP.strip()))
    pump["student"] = 1
    source = "your own orders from the interactive lab"
else:
    pump = simulated_pump_class()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(pump), "orders")
pump_summary = pump.groupby("design").agg(orders=("mode_error", "size"),
                                          mode_errors=("mode_error", "mean"),
                                          wrong_numbers=("value_error", "mean"))
pump_summary["per year, 12 a day"] = expected_errors(pump_summary["mode_errors"],
                                                     12 * 365)
pump_summary["at least one a day"] = p_at_least_one(pump_summary["mode_errors"], 12)
print(pump_summary.round(3).to_string())

**Python note.** The rates are the means of columns of zeros and ones, so `mean` gives the share of orders with a mode error. The last two columns apply the two functions of this section to the measured rates.

In [ ]:
# Figure: mode errors of the two pumps
by_start = pump.groupby(["design", "start_mode"])["mode_error"].mean().unstack()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
names = {"subtle": "small label", "salient": "coloured band"}
labels = [names[d] for d in by_start.index]
x = np.arange(len(by_start))
ax[0].bar(x - 0.2, 100 * by_start.get("rate", 0), 0.4, color="#5A4BB5",
          label="started in rate mode")
ax[0].bar(x + 0.2, 100 * by_start.get("volume", 0), 0.4, color="#C98510",
          label="started in volume mode")
ax[0].set_xticks(x, labels)
ax[0].set_ylabel("orders with a mode error (%)")
ax[0].legend()
n = np.arange(1, 41)
for d, col in zip(pump_summary.index, ["#2E8B57", "#B53F2E"]):
    p_d = float(pump_summary.loc[d, "mode_errors"])
    ax[1].plot(n, p_at_least_one(p_d, n), color=col, label=names[d])
ax[1].set_xlabel("programmings in a day")
ax[1].set_ylabel("at least one mode error")
ax[1].legend()
plt.tight_layout()
plt.show()

**Python note.** `unstack` turns the inner level of the index, the starting mode, into columns. A mode error can only happen when the pump starts in the volume mode, so the left panel shows where the errors come from.

**Observe and reflect.** How many mode errors a year does each pump cause on a ward that programs pumps twelve times a day? Which feature of the second design do you think stopped most of them?

## 4. Defences in layers &#91;[11](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#ref-11)&#93;

*Lecture: [Defences in layers: The Swiss cheese model](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#defences-in-layers)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "pump_summary" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-05/"
               "NB05_heuristics_and_error.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A ward protects patients with several checks in a row, and each catches only part of the errors. Python multiplies the layers to find how many errors reach the patient. It then simulates busy shifts in which all layers weaken at once, a dependence that the simple product ignores.

**From the formulas to the code.** The probability of an error per order $p$ is `p`, and the shares that the layers stop $c_1, c_2, \ldots$ are `catches`. The probability of harm $p \prod (1 - c_i)$ is `p_harm(p, catches)`.

In [ ]:
def p_harm(p, catches):
    """Probability that an error passes every layer of defence."""
    return p * float(np.prod([1 - c for c in catches]))

LAYERS = {"pharmacist review": 0.50, "barcode scan": 0.41, "bedside check": 0.35}
P_ERR = 0.10
print(f"worked example: {1000 * p_harm(P_ERR, LAYERS.values()):.1f} per 1000 orders")
no_scan = [c for k, c in LAYERS.items() if k != "barcode scan"]
print(f"without the scanner: {1000 * p_harm(P_ERR, no_scan):.1f} per 1000 orders")

**Python note.** `np.prod` multiplies all values of a list, here the shares that pass each layer. The dictionary keeps the name of each layer next to its value, so that one layer can be left out by its name.

**Exercise D.** Store the probability of harm per order with an error rate of 0.10 and only the first two layers, the pharmacist (0.50) and the scanner (0.41), in `p_two`.

In [ ]:
p_two = None   # your answer

check("Exercise D", p_two)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_two = p_harm(0.10, [0.50, 0.41])

check("Exercise D", p_two)

In [ ]:
def simulate_shifts(n_shifts=2000, orders=50, busy_share=0.25,
                    seed=RANDOM_STATE):
    """Orders through the layers; busy shifts raise errors and halve catches."""
    rng = np.random.default_rng(seed)
    harm = 0
    for _ in range(n_shifts):
        busy = rng.random() < busy_share
        p = P_ERR * (1.5 if busy else 1.0)
        catches = [c * (0.5 if busy else 1.0) for c in LAYERS.values()]
        errors = rng.random(orders) < p
        for c in catches:
            errors &= rng.random(orders) >= c     # passes this layer
        harm += errors.sum()
    return harm / (n_shifts * orders)

p_avg = P_ERR * (0.75 + 0.25 * 1.5)
c_avg = [c * (0.75 + 0.25 * 0.5) for c in LAYERS.values()]
print(f"simulated with busy shifts: {1000 * simulate_shifts():.1f} per 1000")
print(f"product of the average values: {1000 * p_harm(p_avg, c_avg):.1f} per 1000")

**Python note.** `errors &= rng.random(orders) >= c` keeps an error only if it also passes the next layer. The second line multiplies the average error rate and the average catches, as if the layers failed independently of each other.

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_DRUGS`. Left empty, the variable makes the code use a simulated class of 24 students with assumed error rates.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_DRUGS = """
"""

def simulated_drug_class(n_students=24, seed=RANDOM_STATE):
    """Twelve orders per design; look-alike names cause more wrong picks."""
    rng = np.random.default_rng(seed + 2)
    rows = []
    for student in range(1, n_students + 1):
        for block, (design, p_wrong) in enumerate([("plain", 0.12),
                                                   ("safe", 0.05)], 1):
            for trial in range(1, 13):
                wrong = int(rng.random() < p_wrong)
                caught = wrong if design == "safe" else 0
                rt = rng.lognormal(np.log(2.2 if design == "plain" else 2.4), 0.25)
                rows.append([student, block, design, trial, wrong, caught,
                             round(1000 * rt), 0])
    return pd.DataFrame(rows, columns=["student", "block", "design", "trial",
                                       "wrong", "caught", "rt_ms", "timeout"])

if MY_DRUGS.strip():
    drugs = pd.read_csv(io.StringIO(MY_DRUGS.strip()))
    drugs["student"] = 1
    source = "your own picks from the interactive lab"
else:
    drugs = simulated_drug_class()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(drugs), "orders")
answered = drugs[drugs["timeout"] == 0]
drug_rates = answered.groupby("design").agg(orders=("wrong", "size"),
                                            wrong=("wrong", "mean"),
                                            caught=("caught", "mean"))
drug_rates["reached"] = drug_rates["wrong"] - drug_rates["caught"]
print(drug_rates.round(3))

**Python note.** Orders that ran out of time are left out of the rates, because no drug was picked. The share that reached the patient is the share of wrong picks minus the share that the scanner stopped.

In [ ]:
# Figure: wrong picks and errors that reached the patient
fig, ax = plt.subplots(figsize=(7, 3.4))
x = np.arange(len(drug_rates))
ax.bar(x - 0.2, 100 * drug_rates["wrong"], 0.4, color="#C98510",
       label="wrong first pick")
ax.bar(x + 0.2, 100 * drug_rates["reached"], 0.4, color="#B53F2E",
       label="reached the patient")
names = {"plain": "small letters, side by side",
         "safe": "tall man, apart, scanner"}
ax.set_xticks(x, [names[d] for d in drug_rates.index])
ax.set_ylabel("share of orders (%)")
ax.legend()
plt.tight_layout()
plt.show()

**Python note.** The two bars of each design stand next to each other. The orange one counts the slips of the eye and the hand, and the red one the slips that no layer stopped.

**Observe and reflect.** How much of the improvement comes from the labels and the layout, and how much from the scanner? Which of the two would still work on a night when the scanner is broken?

## 5. The clinic app, stage 5

*Lecture: [The clinic app, stage 5: A heuristic evaluation with a ranked list of problems](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#the-clinic-app-stage-5)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "drug_rates" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-05/"
               "NB05_heuristics_and_error.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Five evaluators inspected the clinic app and found sixteen problems. The stage asks how complete that list is, how many evaluators a second round needs and which problems to fix first. A dictionary keeps every result, and the ranked list goes into the redesign of Week 7.

In [ ]:
STAGE5 = {}
lam_e, gt_e, adj_e = lewis_adjust(M5)
STAGE5["problems found"] = int((M5.sum(axis=0) > 0).sum())
STAGE5["lambda, estimated"] = round(lam_e, 3)
STAGE5["problems found once, share"] = round(gt_e, 3)
STAGE5["lambda, adjusted"] = round(adj_e, 3)
STAGE5["share found, unadjusted"] = round(float(found_share(lam_e, 5)), 3)
STAGE5["share found, adjusted"] = round(float(found_share(adj_e, 5)), 3)
STAGE5["problems in total, estimated"] = round(
    STAGE5["problems found"] / STAGE5["share found, adjusted"], 1)
STAGE5["evaluators for 90 percent"] = evaluators_for(0.90, adj_e)
for key, value in STAGE5.items():
    print(f"{key:<36} {value}")
ranked5 = ev.sort_values(["mean severity", "found by"], ascending=False)
print(ranked5.head(5)[["problem", "description", "mean severity"]]
      .to_string(index=False))

**Python note.** `lewis_adjust(M5)` returns three values, and the first line unpacks them into three names. The estimated total divides the found problems by the share that five evaluators are expected to find.

In [ ]:
n = np.arange(1, 13)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(n, found_share(lam_e, n), "--", color="#B53F2E",
           label=f"unadjusted, lambda = {lam_e:.3f}")
ax[0].plot(n, found_share(adj_e, n), color="#5A4BB5",
           label=f"adjusted, lambda = {adj_e:.3f}")
ax[0].axvline(5, ls=":", color="#585E6E")
ax[0].axhline(0.9, ls=":", color="#585E6E")
ax[0].set_xlabel("number of evaluators")
ax[0].set_ylabel("share of problems found")
ax[0].set_title("How complete is the list?")
ax[0].legend(loc="lower right")
top = ranked5.head(8)
ax[1].barh([f"{p}. {h}" for p, h in zip(top["problem"], top["heuristic"])][::-1],
           top["mean severity"][::-1], color="#5A4BB5")
ax[1].set_xlim(0, 4)
ax[1].set_xlabel("mean severity (0 to 4)")
ax[1].set_title("The eight most severe problems")
plt.tight_layout()
plt.show()

**Python note.** `[::-1]` reverses a list, so that the most severe problem appears at the top of the horizontal bars. The dotted lines mark the five evaluators of the stage and the goal of 90 percent.

**Exercise E.** Store the number of evaluators that the adjusted estimate requires for 90 percent, as kept in `STAGE5`, in `n_round2`.

In [ ]:
n_round2 = None   # your answer

check("Exercise E", n_round2)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_round2 = STAGE5["evaluators for 90 percent"]

check("Exercise E", n_round2)

**Observe and reflect.** Without the adjustment, the five evaluators seem to have found 94 percent of the problems. What would a team decide with that number, and what does the adjusted value change?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE5" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-05/"
               "NB05_heuristics_and_error.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each app below received a heuristic evaluation with a few evaluators. The functions of Section 3 estimate how complete each list is and how many evaluators a second round would need, after one change of a name.

The evaluations below are simulated with illustrative values. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "banking"
# choose: clinic, banking, e_government, food_delivery, course_registration
APPS = {"clinic": (5, 22, 0.30, "Cancel next to Confirm (H5)"),
        "banking": (4, 30, 0.22, "Transfer limits explained only in the PDF (H10)"),
        "e_government": (3, 40, 0.18, "Form codes instead of names (H2)"),
        "food_delivery": (5, 18, 0.35, "No way to undo an order (H3)"),
        "course_registration": (4, 26, 0.25, "Clash of courses shown only at the end (H5)")}

def simulated_evaluation(n_eval, n_problems, lam, seed=RANDOM_STATE):
    """A 0/1 matrix in which each problem has its own detectability."""
    rng = np.random.default_rng(seed)
    p = rng.beta(2, 2 * (1 - lam) / lam, n_problems)
    return (rng.random((n_eval, n_problems)) < p).astype(int)

n_eval, n_problems, lam_true, example = APPS[APP]
M_app = simulated_evaluation(n_eval, n_problems, lam_true)
lam_a, gt_a, adj_a = lewis_adjust(M_app)
found = int((M_app.sum(axis=0) > 0).sum())
print(f"{APP}: {n_eval} evaluators found {found} problems, e.g. {example}")
print(f"  lambda {lam_a:.3f}, adjusted {adj_a:.3f}, share found "
      f"{float(found_share(adj_a, n_eval)):.2f}")
print(f"  evaluators for 90 percent: {evaluators_for(0.90, adj_a)}")

**Python note.** `rng.beta(a, b, n)` draws a detectability between 0 and 1 for each problem, with the mean $a/(a + b)$, here the value `lam` of the app. Some problems are therefore easy to find and others hard, as in real evaluations.

**Observe and reflect.** Which app has the least complete list? Would you spend the money on more evaluators or on a usability test with real users, which Week 9 introduces?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Treat code review as an evaluation: Three reviewers found 9, 7 and 6 of 15 known defects. Build a plausible 0/1 matrix and estimate how many defects remain. |
| Web development | Run a heuristic evaluation of a website you use with two classmates, merge your lists into a matrix and compute the agreement and the adjusted discovery rate. |
| Mobile development | A banking app shows Delete and Save as equal buttons side by side. Estimate the wrong taps per month for 10 000 users with an error rate of 0.5 percent per use. |
| Game development | Map the controls of a game onto a gamepad in two ways and argue with Norman's principles which mapping produces fewer errors. |
| Embedded systems and the Internet of Things | A smart oven has four rings and four knobs in a row. Use the error rates of the stove study to estimate the wrong rings per year of a family. |
| Data science | Simulate evaluations with `simulated_evaluation` for 2 to 10 evaluators and show how far the unadjusted estimate of lambda overestimates the true value. |
| Cybersecurity | A security warning is confirmed with OK hundreds of times a year. Model it as a layer whose catch falls with each repetition and compute the harm over a year. |
| Health informatics | Add a fourth layer, a pharmacist on the ward who stops 30 percent of the remaining errors, and compute the harm per 1000 orders with and without the scanner. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-05).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>